# NB20 — ECE Tie-Degeneracy Fix, Corrected Results, and Undecided Sections

Consequence of the post-referee audit: equal-mass ECE with a stable sort fabricates calibration error on tie-heavy predictions (isotonic outputs, low-signal regimes), which fabricated the oracle competence-calibration coupling. This notebook:

1. **Patches `src/metrics.py`** (tie-safe equal-mass binning, seeded) + regression test.
2. **Regenerates the corrected canonical CSVs in-repo**: per-generator oracle-corrected ECE (hybrid/Platt/beta, paper-estimator vs tie-safe) for all three detectors; raw tie-safe ECE; transferred-calibrator tie-safe ECE (iso/Platt/beta); orientation audit.
3. **Equity, tie-safe** (Section 5's fate): per-subgroup ECE, Table 5 worst-vs-pooled gaps and Table 6 pairwise gaps with identity-clustered pivotal CIs.
4. **Competence-adjusted subgroup regression** (reviewer requirement): cell-level ECE ~ AUC + axis indicators, identity-clustered bootstrap.
5. **Figure 9 retargeted** to transferred ECE (the deployable risk) + gen-level bootstrap CI on monitor ROC-AUC; **mediation paired-difference bootstrap** on transferred-ECE structure.
6. Commit ritual.

Everything is reproduced from per-frame scores; committed numbers are asserted before new ones are computed.

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, sys, subprocess
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
PARENT = "/content/drive/MyDrive/CDTS_Research"
for f in [".gitconfig",".git-credentials"]:
    if os.path.exists(f"{PARENT}/{f}"): subprocess.run(f'cp "{PARENT}/{f}" /root/{f}', shell=True)
subprocess.run("git config --global credential.helper store", shell=True)
os.chdir(REPO)
for k in list(sys.modules.keys()):
    if k in ("metrics","calibration"): del sys.modules[k]
sys.path = [p for p in sys.path if "DeepfakeBench" not in p]
sys.path.insert(0, f"{REPO}/src")
import pandas as pd, numpy as np
CALD = f"{REPO}/reports/calibration"; SCOR = f"{REPO}/reports/scores"
print("repo:", os.getcwd())

Mounted at /content/drive
repo: /content/drive/MyDrive/CDTS_Research/deepfake-trust-research


## Part 1 — Patch metrics.py (tie-safe equal-mass binning) + regression test

In [2]:
# Patch: _bin_indices gains tie-safe behaviour via infinitesimal seeded jitter before the sort.
# The old behaviour is kept available as scheme='equal_mass_legacy' for exact reproduction of v1 numbers.
src_path = f"{REPO}/src/metrics.py"
src = open(src_path).read()

OLD = '''    if scheme == "equal_mass":
        order = np.argsort(p, kind="mergesort")
        return [b for b in np.array_split(order, n_bins) if b.size > 0]'''
NEW = '''    if scheme == "equal_mass":
        # tie-safe: break ties randomly (seeded) so bins do not follow input row order.
        # Stable-sort tie handling is degenerate on tie-heavy predictions (e.g. isotonic
        # outputs): with label-ordered rows a CONSTANT predictor is assigned ECE ~ 2p(1-p)
        # instead of ~|prevalence shift|. See NB20 regression test.
        _rng = np.random.default_rng(0)
        order = np.argsort(p + _rng.random(p.size) * 1e-12, kind="mergesort")
        return [b for b in np.array_split(order, n_bins) if b.size > 0]
    elif scheme == "equal_mass_legacy":
        order = np.argsort(p, kind="mergesort")
        return [b for b in np.array_split(order, n_bins) if b.size > 0]'''

if 'equal_mass_legacy' in src:
    print("metrics.py already patched - skipping")
else:
    assert OLD in src, "expected _bin_indices body not found; inspect src/metrics.py manually"
    open(src_path, 'w').write(src.replace(OLD, NEW))
    print("metrics.py patched")

for k in list(sys.modules.keys()):
    if k in ("metrics","calibration"): del sys.modules[k]
import metrics as met, calibration as cal

# --- regression test: constant predictor on label-sorted rows ---
y = np.array([0]*1700 + [1]*8300)          # label-sorted, prevalence .83
p = np.full_like(y, 0.838, dtype=float)     # constant prediction near prevalence
e_new = met.ece(p, y, 15, 'equal_mass')
e_old = met.ece(p, y, 15, 'equal_mass_legacy')
print(f"constant predictor: tie-safe ECE = {e_new:.4f} (must be ~= {abs(0.838-0.83):.4f}); legacy = {e_old:.4f} (the artifact)")
assert e_new < 0.02 and e_old > 0.2, "regression test failed"
print("REGRESSION TEST PASSED")

metrics.py patched
constant predictor: tie-safe ECE = 0.0112 (must be ~= 0.0080); legacy = 0.2672 (the artifact)
REGRESSION TEST PASSED


## Part 2 — Regenerate corrected canonical CSVs (all three detectors, from per-frame scores)

In [3]:
from sklearn.metrics import roc_auc_score
from scipy.stats import pearsonr

def split_gen(df):
    p = df.prob_fake.values.astype(float); y = df.label.values.astype(int)
    ci, ti, _ = cal.leakage_safe_split(y, groups=df.identity_id.astype(str).values, calib_frac=0.5, seed=42)
    return p[ci], y[ci], p[ti], y[ti]

def E(v, yt): return met.ece(np.clip(v,0,1), yt, 15, 'equal_mass')
def E_legacy(v, yt): return met.ece(np.clip(v,0,1), yt, 15, 'equal_mass_legacy')

DET = {'xception': ('xceptionFS','labelfree_signals.csv'),
       'effnetb4': ('effnetb4','unified_trust_signals_effnet.csv'),
       'clip': ('clip','unified_trust_signals_clip.csv')}
rows = []
for det,(key,csvf) in DET.items():
    for gen in pd.read_csv(f"{CALD}/{csvf}").method.tolist():
        df = pd.read_parquet(f"{SCOR}/{key}_df40_{gen}.parquet")
        pc,yc,pt,yt = split_gen(df)
        ph,_ = cal.fit_predict("hybrid", pc, yc, pt, switch_threshold_n=1000)
        ppl = cal.PlattScaling().fit(pc,yc).predict(pt)
        pbe = cal.BetaCalibration().fit(pc,yc).predict(pt)
        rows.append(dict(detector=det, method=gen, AUC=roc_auc_score(yt,pt),
                         ECE_raw=E(pt,yt),
                         ECE_hybrid_legacy=E_legacy(ph,yt), ECE_hybrid=E(ph,yt),
                         ECE_platt=E(ppl,yt), ECE_beta=E(pbe,yt)))
        print(f"{det:9s} {gen:12s} AUC {rows[-1]['AUC']:.3f} raw {rows[-1]['ECE_raw']:.3f} "
              f"hyb {rows[-1]['ECE_hybrid_legacy']:.3f}->{rows[-1]['ECE_hybrid']:.3f} beta {rows[-1]['ECE_beta']:.3f}")
G = pd.DataFrame(rows)
G.to_csv(f"{CALD}/oracle_corrected_pergen.csv", index=False)
print()
for det in DET:
    d = G[G.detector==det]
    for col in ['ECE_raw','ECE_hybrid_legacy','ECE_hybrid','ECE_beta']:
        r,p = pearsonr(d.AUC, d[col]); print(f"{det:9s} {col:18s} r={r:+.3f} p={p:.1e}")

# transferred calibrator (Xception, in-domain FS pool, frozen)
INDOM = ['simswap','blendface','facedancer','fsgan','faceswap','inswap']
gens21 = pd.read_csv(f"{CALD}/labelfree_signals.csv").method.tolist()
pcs,ycs,splits = [],[],{}
for gen in gens21:
    df = pd.read_parquet(f"{SCOR}/xceptionFS_df40_{gen}.parquet")
    pc,yc,pt,yt = split_gen(df); splits[gen]=(pt,yt)
    if gen in INDOM: pcs.append(pc); ycs.append(yc)
pc_pool, yc_pool = np.concatenate(pcs), np.concatenate(ycs)
tr = {'iso': cal.IsotonicCalibration().fit(pc_pool,yc_pool),
      'platt': cal.PlattScaling().fit(pc_pool,yc_pool),
      'beta': cal.BetaCalibration().fit(pc_pool,yc_pool)}
trows = []
for gen in gens21:
    pt,yt = splits[gen]
    trows.append(dict(method=gen, AUC=roc_auc_score(yt,pt), ECE_raw=E(pt,yt),
                      **{f'ECE_transf_{k}': E(c.predict(pt),yt) for k,c in tr.items()}))
T = pd.DataFrame(trows); T.to_csv(f"{CALD}/transferred_calibrator_tiesafe.csv", index=False)
for k in tr:
    r,p = pearsonr(T.AUC, T[f'ECE_transf_{k}'])
    out = T[~T.method.isin(INDOM)]; r2,p2 = pearsonr(out.AUC, out[f'ECE_transf_{k}'])
    print(f"transferred-{k:5s} r={r:+.3f} (all 21), r={r2:+.3f} (15 OOD)")
print("saved oracle_corrected_pergen.csv + transferred_calibrator_tiesafe.csv")

xception  faceswap     AUC 0.892 raw 0.069 hyb 0.045->0.018 beta 0.018
xception  fomm         AUC 0.801 raw 0.156 hyb 0.054->0.018 beta 0.014
xception  fsgan        AUC 0.900 raw 0.107 hyb 0.036->0.022 beta 0.027
xception  wav2lip      AUC 0.470 raw 0.511 hyb 0.224->0.025 beta 0.018
xception  StyleGAN2    AUC 0.649 raw 0.326 hyb 0.063->0.014 beta 0.019
xception  ddim         AUC 0.766 raw 0.197 hyb 0.041->0.013 beta 0.018
xception  simswap      AUC 0.948 raw 0.083 hyb 0.024->0.019 beta 0.020
xception  StyleGAN3    AUC 0.679 raw 0.289 hyb 0.060->0.013 beta 0.016
xception  facevid2vid  AUC 0.822 raw 0.134 hyb 0.064->0.037 beta 0.040
xception  pirender     AUC 0.688 raw 0.265 hyb 0.089->0.044 beta 0.045
xception  DiT          AUC 0.521 raw 0.475 hyb 0.161->0.011 beta 0.015
xception  StyleGANXL   AUC 0.637 raw 0.341 hyb 0.093->0.016 beta 0.026
xception  lia          AUC 0.607 raw 0.360 hyb 0.146->0.034 beta 0.035
xception  sd2.1        AUC 0.688 raw 0.272 hyb 0.044->0.015 beta 0.013
xcepti

## Part 3 — Equity under the tie-safe estimator (Section 5's fate)

Reproduces the NB07 join and split exactly, asserts legacy per-subgroup ECEs match the committed table, then recomputes everything tie-safe: per-subgroup ECE, Table 5 worst-vs-pooled gaps, Table 6 pairwise gaps, all with identity-clustered pivotal 95% CIs (B = 2000).

In [4]:
import re
XU = f"{REPO}/data/annotations/Xu_DeepFakeAnnotations"
DEMO = ['male','young','middle_aged','senior','asian','white','black','shiny_skin']
ff = pd.read_csv(f"{XU}/A-FF++.csv", usecols=['path','label']+DEMO)
ff['ident'] = ff['path'].map(lambda p: (re.search(r'face_images/(\d+)(?:_\d+)?/', p) or [None,None])[1] if isinstance(p,str) else None)
def majc(s):
    conf = s[s != 0]
    return np.nan if len(conf)==0 else (1 if (conf==1).sum() >= (conf==-1).sum() else -1)
ident_demo = ff.groupby('ident')[DEMO].agg(majc).reset_index()
sc = pd.read_parquet(f"{SCOR}/xception_ffpp_test.parquet")
sc['ident'] = sc['identity_id'].astype(str).str.zfill(3)
j = sc.merge(ident_demo, on='ident', how='inner')
p = j.prob_fake.values.astype(float); y = j.label.values.astype(int)
ci, ti, _ = cal.leakage_safe_split(y, groups=j.ident.values, calib_frac=0.5, seed=42)
p_cal, _ = cal.fit_predict("hybrid", p[ci], y[ci], p[ti], switch_threshold_n=1000)
ev = j.iloc[ti].copy(); ev['p_cal'] = p_cal

committed = pd.read_csv(f"{CALD}/equity_subgroup_ece_xceptionFS.csv")
def grp(axis, gname, d):
    if axis in ('age','ethnicity'): return d[d[gname]==1]
    col,val = gname.split('='); return d[d[col]==(1 if val=='1' else -1)]
print("subgroup      n      legacy(committed)  tie-safe")
new_rows = []
for _, r in committed.iterrows():
    g = grp(r.axis, r.group, ev)
    e_leg = met.ece(g.p_cal.values, g.label.values, 15, 'equal_mass_legacy')
    e_new = met.ece(g.p_cal.values, g.label.values, 15, 'equal_mass')
    ok = abs(e_leg - r.ECE) < 2e-3
    print(f"{r.group:14s} {len(g):5d}  {e_leg:.4f} ({r.ECE:.4f}) {'OK ' if ok else 'MISMATCH'}  {e_new:.4f}")
    assert ok, f"legacy reproduction failed for {r.group}"
    new_rows.append(dict(axis=r.axis, group=r.group, n=len(g), ECE_legacy=e_leg, ECE_tiesafe=e_new))
pd.DataFrame(new_rows).to_csv(f"{CALD}/equity_subgroup_ece_tiesafe.csv", index=False)

# Table 5 + Table 6 gaps, tie-safe, clustered pivotal CIs
AXES = {'gender':['male'], 'age':['young','middle_aged','senior'],
        'ethnicity':['asian','white','black'], 'skintone':['shiny_skin']}
from sklearn.metrics import roc_auc_score as auc_fn
def worst_gap(d, axis, cols):
    pooled = met.ece(d.p_cal.values, d.label.values, 15, 'equal_mass')
    eces = []
    if axis in ('age','ethnicity'):
        for a in cols:
            g = d[d[a]==1]
            if len(g)>=30: eces.append(met.ece(g.p_cal.values, g.label.values, 15, 'equal_mass'))
    else:
        for v in (1,-1):
            g = d[d[cols[0]]==v]
            if len(g)>=30: eces.append(met.ece(g.p_cal.values, g.label.values, 15, 'equal_mass'))
    return (max(eces)-pooled) if eces else np.nan
PAIRS = {'gender (M vs F)': lambda d: (d[d.male==1], d[d.male==-1]),
         'skin appearance': lambda d: (d[d.shiny_skin==1], d[d.shiny_skin==-1]),
         'age (young vs senior)': lambda d: (d[d.young==1], d[d.senior==1])}
def pair_gaps(d, fn):
    g1,g2 = fn(d)
    if min(len(g1),len(g2))<30 or g1.label.nunique()<2 or g2.label.nunique()<2: return np.nan, np.nan
    return (auc_fn(g1.label,g1.p_cal)-auc_fn(g2.label,g2.p_cal),
            met.ece(g1.p_cal.values,g1.label.values,15,'equal_mass')-met.ece(g2.p_cal.values,g2.label.values,15,'equal_mass'))

B = 2000; rng = np.random.default_rng(42)
ig = ev.groupby('ident').indices; idents = list(ig.keys()); n_id = len(idents)
boot_idx = [np.concatenate([ig[i] for i in rng.choice(idents, size=n_id, replace=True)]) for _ in range(B)]
print(f"\nclustered bootstrap: {n_id} identities, B={B}")
print("\n=== Table 5 (worst-vs-pooled), tie-safe ===")
t5 = []
for axis, cols in AXES.items():
    pt = worst_gap(ev, axis, cols)
    boots = np.array([worst_gap(ev.iloc[bi], axis, cols) for bi in boot_idx])
    boots = boots[~np.isnan(boots)]
    lo,hi = 2*pt-np.percentile(boots,97.5), 2*pt-np.percentile(boots,2.5)
    t5.append(dict(axis=axis, gap=round(pt,4), ci_lo=round(lo,4), ci_hi=round(hi,4), excludes_0=lo>0))
    print(f"  {axis:10s} gap={pt:+.4f} CI [{lo:+.4f},{hi:+.4f}] {'EXCLUDES 0' if lo>0 else 'includes 0'}")
pd.DataFrame(t5).to_csv(f"{CALD}/equity_gap_CIs_tiesafe.csv", index=False)
print("\n=== Table 6 (pairwise), tie-safe ===")
t6 = []
for name, fn in PAIRS.items():
    pa, pe = pair_gaps(ev, fn)
    ba, be = [], []
    for bi in boot_idx:
        a,e = pair_gaps(ev.iloc[bi], fn)
        if not np.isnan(a): ba.append(a)
        if not np.isnan(e): be.append(e)
    for metr, pt_, boots in [('AUC_gap',pa,np.array(ba)), ('ECE_gap',pe,np.array(be))]:
        lo,hi = 2*pt_-np.percentile(boots,97.5), 2*pt_-np.percentile(boots,2.5)
        ex = lo>0 or hi<0
        t6.append(dict(pair=name, metric=metr, point=round(pt_,4), ci_lo=round(lo,4), ci_hi=round(hi,4), excludes_0=ex))
        print(f"  {name:22s} {metr:8s} {pt_:+.4f} CI [{lo:+.4f},{hi:+.4f}] {'EXCLUDES 0' if ex else 'includes 0'}")
pd.DataFrame(t6).to_csv(f"{CALD}/equity_pairwise_gap_CIs_tiesafe.csv", index=False)

subgroup      n      legacy(committed)  tie-safe
male=1          5117  0.0735 (0.0735) OK   0.0202
male=0          6073  0.0642 (0.0642) OK   0.0331
young           5755  0.0608 (0.0608) OK   0.0335
middle_aged      959  0.1148 (0.1148) OK   0.0662
senior          1120  0.0754 (0.0754) OK   0.0437
asian            639  0.0610 (0.0610) OK   0.0510
white           9273  0.0340 (0.0340) OK   0.0191
shiny_skin=1    5753  0.0769 (0.0769) OK   0.0318
shiny_skin=0    2560  0.0618 (0.0618) OK   0.0264

clustered bootstrap: 70 identities, B=2000

=== Table 5 (worst-vs-pooled), tie-safe ===
  gender     gap=+0.0119 CI [-0.0060,+0.0189] includes 0
  age        gap=+0.0450 CI [-0.0182,+0.0623] includes 0
  ethnicity  gap=+0.0298 CI [-0.0961,+0.0527] includes 0
  skintone   gap=+0.0106 CI [-0.0149,+0.0175] includes 0

=== Table 6 (pairwise), tie-safe ===
  gender (M vs F)        AUC_gap  -0.0061 CI [-0.0476,+0.0316] includes 0
  gender (M vs F)        ECE_gap  -0.0129 CI [-0.0391,+0.0019] includes 

## Part 4 — Competence-adjusted subgroup regression (reviewer requirement)

Cells = subgroup x FF++ forgery method. Model: cell ECE ~ cell AUC + axis indicators, identity-clustered bootstrap on the coefficients. Answers whether subgroup competence explains subgroup calibration or whether subgroup membership retains an independent association.

In [5]:
GROUPS = [('gender','male=1'),('gender','male=0'),('age','young'),('age','middle_aged'),('age','senior'),
          ('ethnicity','asian'),('ethnicity','white'),('skintone','shiny_skin=1'),('skintone','shiny_skin=0')]
METHODS = ['faceswap','deepfakes','face2face','neuraltextures']
reals = ev[ev.label==0]
def build_cells(d):
    dre = d[d.label==0]
    out = []
    for m in METHODS:
        dm = pd.concat([dre, d[(d.label==1)&(d.method==m)]])
        for axis, gname in GROUPS:
            g = grp(axis, gname, dm)
            if len(g)>=60 and g.label.nunique()==2:
                out.append(dict(method=m, axis=axis, group=gname, n=len(g),
                                AUC=auc_fn(g.label, g.p_cal),
                                ECE=met.ece(g.p_cal.values, g.label.values, 15, 'equal_mass')))
    return pd.DataFrame(out)
cells_df = build_cells(ev)
print(f"cells: {len(cells_df)}")
import statsmodels.formula.api as smf
def fit_coefs(cdf):
    m = smf.ols('ECE ~ AUC + C(axis)', data=cdf).fit()
    return m.params
base = fit_coefs(cells_df)
print("\nOLS on cells (point estimates):\n", base.round(4).to_string())
coef_boot = {k: [] for k in base.index}
for bi in boot_idx[:1000]:
    try:
        cb = build_cells(ev.iloc[bi])
        pb = fit_coefs(cb)
        for k in coef_boot:
            if k in pb: coef_boot[k].append(pb[k])
    except Exception:
        pass
print("\nidentity-clustered 95% CIs (1000 resamples):")
for k, v in coef_boot.items():
    v = np.array(v); lo,hi = np.percentile(v,[2.5,97.5])
    print(f"  {k:24s} {base[k]:+.4f}  [{lo:+.4f},{hi:+.4f}]  {'EXCLUDES 0' if lo>0 or hi<0 else 'includes 0'}")
cells_df.to_csv(f"{CALD}/equity_competence_adjusted_cells.csv", index=False)

cells: 36

OLS on cells (point estimates):
 Intercept               0.0745
C(axis)[T.ethnicity]    0.0070
C(axis)[T.gender]      -0.0052
C(axis)[T.skintone]    -0.0142
AUC                     0.2710

identity-clustered 95% CIs (1000 resamples):
  Intercept                +0.0745  [+0.0365,+0.2035]  EXCLUDES 0
  C(axis)[T.ethnicity]     +0.0070  [-0.0294,+0.0749]  includes 0
  C(axis)[T.gender]        -0.0052  [-0.0374,+0.0132]  includes 0
  C(axis)[T.skintone]      -0.0142  [-0.0480,+0.0078]  includes 0
  AUC                      +0.2710  [+0.0934,+0.2983]  EXCLUDES 0


## Part 5 — Figure 9 retargeted to transferred ECE + mediation paired-difference bootstrap

In [ ]:
from sklearn.metrics import roc_auc_score as roc
T = pd.read_csv(f"{CALD}/transferred_calibrator_tiesafe.csv")
L = pd.read_csv(f"{CALD}/labelfree_signals.csv")[['method','entropy','ks_vs_ref','std_score']]
M = T.merge(L, on='method')
ylab = (M.ECE_transf_iso > 0.1).astype(int)
print(f"transferred-ECE > 0.1: {ylab.sum()} of {len(M)} generators: {sorted(M[ylab==1].method)}")
rng = np.random.default_rng(42)
for sig, direction in [('entropy',1), ('ks_vs_ref',-1), ('std_score',1)]:
    a = roc(ylab, direction*M[sig])
    bs = []
    for _ in range(5000):
        i = rng.integers(0, len(M), len(M))
        if ylab.values[i].min()==ylab.values[i].max(): continue
        bs.append(roc(ylab.values[i], (direction*M[sig]).values[i]))
    lo,hi = np.percentile(bs,[2.5,97.5])
    print(f"  monitor {sig:10s} ROC-AUC = {a:.3f} [{lo:.3f},{hi:.3f}]")

# mediation as paired differences: bootstrap Delta = r_raw - r_partial over generators
U = pd.read_csv(f"{CALD}/unified_trust_signals.csv")
MM = T.merge(U[['method','faithfulness_rankcorr','ks_vs_ref']], on='method')
def resid(y, x):
    b = np.polyfit(x, y, 1); return y - (b[0]*x + b[1])
def raw_partial(d, a, b):
    from scipy.stats import pearsonr as pr
    r_raw = pr(d[a], d[b])[0]
    r_par = pr(resid(d[a].values, d.AUC.values), resid(d[b].values, d.AUC.values))[0]
    return r_raw, r_par
pairs = [('ECE_transf_iso','faithfulness_rankcorr'), ('ECE_transf_iso','ks_vs_ref'), ('faithfulness_rankcorr','ks_vs_ref')]
print("\nmediation, paired-difference bootstrap (Delta = |r_raw| - |r_partial|, 5000 gen resamples):")
for a,b in pairs:
    r0, p0 = raw_partial(MM, a, b)
    ds = []
    for _ in range(5000):
        i = rng.integers(0, len(MM), len(MM))
        d = MM.iloc[i]
        if d.AUC.std()<1e-9: continue
        rr, rp = raw_partial(d, a, b)
        ds.append(abs(rr)-abs(rp))
    lo,hi = np.percentile(ds,[2.5,97.5])
    print(f"  {a} ~ {b}: raw {r0:+.3f}, partial {p0:+.3f}, Delta CI [{lo:+.3f},{hi:+.3f}] {'EXCLUDES 0' if lo>0 else 'includes 0'}")

transferred-ECE > 0.1: 14 of 21 generators: ['DiT', 'SiT', 'StyleGAN2', 'StyleGAN3', 'StyleGANXL', 'ddim', 'fomm', 'lia', 'pirender', 'pixart', 'rddm', 'sadtalker', 'sd2.1', 'wav2lip']
  monitor entropy    ROC-AUC = 0.990 [0.941,1.000]
  monitor ks_vs_ref  ROC-AUC = 0.939 [0.808,1.000]


## Part 6 — Commit ritual

In [ ]:
import subprocess, os
os.chdir(REPO)
files = ["src/metrics.py",
         "reports/calibration/oracle_corrected_pergen.csv",
         "reports/calibration/transferred_calibrator_tiesafe.csv",
         "reports/calibration/equity_subgroup_ece_tiesafe.csv",
         "reports/calibration/equity_gap_CIs_tiesafe.csv",
         "reports/calibration/equity_pairwise_gap_CIs_tiesafe.csv",
         "reports/calibration/equity_competence_adjusted_cells.csv",
         "notebooks/NB20_ece_fix_and_corrected_results.ipynb"]
subprocess.run("git add " + " ".join(files), shell=True)
print(subprocess.run("git status --short", shell=True, capture_output=True, text=True).stdout)
msg = ("NB20: fix ECE tie-degeneracy (tie-safe equal-mass binning, legacy kept for reproduction) "
       "+ regression test; regenerate corrected per-generator, transferred-calibrator, and equity "
       "results; competence-adjusted subgroup regression; Figure 9 retargeted to transferred ECE")
subprocess.run(f'git commit -m "{msg}"', shell=True)
r = subprocess.run("git push", shell=True, capture_output=True, text=True)
print(r.stdout, r.stderr)
print(subprocess.run("git log --oneline -2", shell=True, capture_output=True, text=True).stdout)